In [ ]:
import subprocess
import shutil
import os
import time
import random
from pathlib import Path

# ============================================================
# Santa 2025 / bbox3 runner
#  - BEST-KEEPING + ROLLBACK (only accept improvements)
#  - Reject overlaps (invalid)
#  - Bias search toward N with largest current contribution
#  - TIME LIMIT stop (default ~3000s)
#  - Log + periodic checkpoints
# ============================================================

# --- 1) Paths (Kaggle defaults + local fallbacks) ---
INPUT_SUB = '/kaggle/input/santa-submission/submission.csv'
INPUT_BIN = '/kaggle/input/a-bit-better/bbox3'

WORKING_DIR = '/kaggle/working/'
WORKING_DIR = str(Path(WORKING_DIR))

# Local fallback (for running outside Kaggle)
if not os.path.exists(INPUT_SUB):
    INPUT_SUB = str(Path.cwd() / 'submission.csv')
if not os.path.exists(INPUT_BIN):
    # If you uploaded bbox3 into the notebook environment, you can point here.
    INPUT_BIN = str(Path('/mnt/data/bbox3')) if os.path.exists('/mnt/data/bbox3') else INPUT_BIN

print("📂 Setting up environment...")
os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)

# Copy submission to working directory
if os.path.exists(INPUT_SUB):
    shutil.copy(INPUT_SUB, os.path.join(WORKING_DIR, 'submission.csv'))
    print(f"✅ Copied {INPUT_SUB} -> {os.path.join(WORKING_DIR, 'submission.csv')}")
else:
    raise FileNotFoundError(f"❌ submission.csv not found at {INPUT_SUB}")

# Copy bbox3 binary to working directory
if os.path.exists(INPUT_BIN):
    shutil.copy(INPUT_BIN, os.path.join(WORKING_DIR, 'bbox3'))
    print(f"✅ Copied {INPUT_BIN} -> {os.path.join(WORKING_DIR, 'bbox3')}")
else:
    raise FileNotFoundError(f"❌ bbox3 not found at {INPUT_BIN}")

# Make bbox3 executable
subprocess.run(["chmod", "+x", "./bbox3"], check=False)

# --- 2) Geometry / scoring helpers ---
import pandas as pd
import numpy as np
from shapely.geometry import Polygon
from shapely.affinity import rotate, translate
from shapely.strtree import STRtree
from decimal import Decimal

class ChristmasTree:
    def __init__(self, center_x, center_y, angle_deg, scale_factor=1e18):
        self.center_x = Decimal(str(center_x))
        self.center_y = Decimal(str(center_y))
        self.angle_deg = Decimal(str(angle_deg))
        self.scale_factor = Decimal(str(scale_factor))

        # Define a "tree" polygon (template)
        trunk_width = Decimal("0.15")
        trunk_height = Decimal("0.2")
        base_width = Decimal("0.7")
        middle_width = Decimal("0.4")
        top_width = Decimal("0.25")

        tip_height = Decimal("0.8")
        top_layer = Decimal("0.5")
        middle_layer = Decimal("0.25")
        base_layer = Decimal("0.0")

        trunk_bottom = base_layer - trunk_height

        coords = [
            (Decimal("0"), tip_height),
            (top_width / 2, top_layer),
            (top_width / 4, top_layer),
            (middle_width / 2, middle_layer),
            (middle_width / 4, middle_layer),
            (base_width / 2, base_layer),
            (trunk_width / 2, base_layer),
            (trunk_width / 2, trunk_bottom),
            (-trunk_width / 2, trunk_bottom),
            (-trunk_width / 2, base_layer),
            (-base_width / 2, base_layer),
            (-middle_width / 4, middle_layer),
            (-middle_width / 2, middle_layer),
            (-top_width / 4, top_layer),
            (-top_width / 2, top_layer),
        ]

        scaled_coords = [(float(x * self.scale_factor), float(y * self.scale_factor)) for x, y in coords]
        poly = Polygon(scaled_coords)

        # Rotate around origin then translate
        poly = rotate(poly, float(self.angle_deg), origin=(0, 0), use_radians=False)
        poly = translate(poly, float(self.center_x * self.scale_factor), float(self.center_y * self.scale_factor))

        self.poly = poly

def load_configuration_from_df(n, df):
    prefix = f"{n:03d}_"
    group = df[df["id"].astype(str).str.startswith(prefix)]
    trees = []
    for _, row in group.iterrows():
        x_str = str(row["x"])
        y_str = str(row["y"])
        d_str = str(row["deg"])
        x = float(x_str[1:]) if x_str.startswith("s") else float(x_str)
        y = float(y_str[1:]) if y_str.startswith("s") else float(y_str)
        deg = float(d_str[1:]) if d_str.startswith("s") else float(d_str)
        trees.append(ChristmasTree(x, y, deg))
    return trees

def get_group_square_side(trees, scale_factor=1e18):
    minx = min(p.bounds[0] for p in (t.poly for t in trees))
    miny = min(p.bounds[1] for p in (t.poly for t in trees))
    maxx = max(p.bounds[2] for p in (t.poly for t in trees))
    maxy = max(p.bounds[3] for p in (t.poly for t in trees))
    w = (maxx - minx) / scale_factor
    h = (maxy - miny) / scale_factor
    return float(max(w, h))

def get_group_score(trees, n):
    s = get_group_square_side(trees)
    return (s * s) / n

def has_overlap(trees):
    """Shapely 1.x/2.x compatible overlap check."""
    polygons = [t.poly for t in trees]
    tree = STRtree(polygons)

    # Shapely 2.x: STRtree.query returns indices; Shapely 1.x returns geometries
    def _returns_indices(arr):
        if arr is None or len(arr) == 0:
            return False
        return isinstance(arr[0], (int, np.integer))

    test = tree.query(polygons[0])
    returns_idx = _returns_indices(test)

    for i, poly in enumerate(polygons):
        try:
            cand = tree.query(poly, predicate="intersects")  # Shapely 2.x fast path
        except TypeError:
            cand = tree.query(poly)

        if returns_idx:
            for j in cand:
                j = int(j)
                if j <= i:
                    continue
                other = polygons[j]
                if poly.intersects(other) and not poly.touches(other):
                    return True
        else:
            for other in cand:
                if other is poly:
                    continue
                if poly.intersects(other) and not poly.touches(other):
                    return True
    return False

def eval_df_sub(df, verb=False, return_details=False):
    total = 0.0
    failed = []
    per_n = {}

    for n in range(1, 201):
        trees = load_configuration_from_df(n, df)
        score = get_group_score(trees, n)
        per_n[n] = score
        total += score
        if verb:
            print(f"{n:3}  {score:.12f}")
        if has_overlap(trees):
            failed.append(n)

    if verb:
        if len(failed) == 0:
            print("✅ Overlap check: OK")
        else:
            print("❌ Overlap check failed for", *failed)
        print(f"Total score: {total:.12f}")

    if return_details:
        return total, per_n, failed
    return total

# --- 3) bbox3 wrapper ---
def run_bbox3(input_csv: str, output_csv: str, n: int, r: int, timeout_s: int = 1200) -> int:
    """Try bbox3 with -i/-o first; fallback to in-place mode."""
    cmd_io = ["./bbox3", "-i", input_csv, "-o", output_csv, "-n", str(n), "-r", str(r)]
    try:
        p = subprocess.run(cmd_io, timeout=timeout_s)
        if p.returncode == 0 and os.path.exists(output_csv):
            return 0
    except Exception:
        pass

    cmd_inplace = ["./bbox3", "-n", str(n), "-r", str(r)]
    p = subprocess.run(cmd_inplace, timeout=timeout_s)
    if p.returncode == 0 and os.path.exists("submission.csv"):
        shutil.copy("submission.csv", output_csv)
    return p.returncode

# --- 4) Search strategy knobs ---
MAX_ITERS = 300            # number of attempts
TIME_LIMIT_S = 3000        # wall-clock time limit (seconds)
TOPK_N = 40                # pick N from the top-K contributors
EPS_IMPROVE = 1e-12        # accept only if strictly better by this margin
CHECKPOINT_EVERY = 25      # save best checkpoint every K iters

R_CANDIDATES = [8, 12, 16, 20, 26, 32, 40, 50]

def pick_target_n(per_n: dict, topk: int = 40):
    items = sorted(per_n.items(), key=lambda kv: kv[1], reverse=True)
    top = items[:topk]
    ns = [n for n, _ in top]
    weights = np.array([s for _, s in top], dtype=float)
    weights = weights / weights.sum() if weights.sum() > 0 else None
    return int(np.random.choice(ns, p=weights))

def pick_r(iter_idx: int):
    # Simple annealing-like schedule
    if iter_idx < MAX_ITERS * 0.25:
        pool = R_CANDIDATES[-4:]
    elif iter_idx < MAX_ITERS * 0.70:
        pool = R_CANDIDATES[2:]
    else:
        pool = R_CANDIDATES[:5]
    return int(random.choice(pool))

# --- 5) BEST-KEEPING loop ---
def main():
    random.seed()

    base_csv = "submission.csv"
    best_csv = "best_submission.csv"
    trial_csv = "trial_submission.csv"
    log_csv = "bbox3_search_log.csv"

    df0 = pd.read_csv(base_csv)
    best_score, per_n, failed = eval_df_sub(df0, verb=True, return_details=True)
    if failed:
        print("⚠️ WARNING: Your starting submission has overlaps for N:", failed)
    shutil.copy(base_csv, best_csv)

    print("\n🚀 Starting best-keeping search...")
    rows = []
    t0 = time.time()
    deadline = t0 + TIME_LIMIT_S

    for it in range(1, MAX_ITERS + 1):
        if time.time() >= deadline:
            print(f"\n⏱️ Time limit reached ({TIME_LIMIT_S}s). Saving current best and stopping...")
            break

        target_n = pick_target_n(per_n, TOPK_N)
        r = pick_r(it)

        # Always start from current best
        shutil.copy(best_csv, base_csv)

        rc = run_bbox3(input_csv=base_csv, output_csv=trial_csv, n=target_n, r=r, timeout_s=1200)
        if rc != 0 or (not os.path.exists(trial_csv)):
            rows.append({"iter": it, "n": target_n, "r": r, "rc": rc, "accepted": 0, "best_score": best_score, "trial_score": None, "fail_n": "bbox3_error"})
            continue

        df_try = pd.read_csv(trial_csv)
        trial_score, per_n_try, failed_try = eval_df_sub(df_try, verb=False, return_details=True)

        accepted = 0
        reason = ""
        if failed_try:
            reason = f"overlap:{','.join(map(str, failed_try[:10]))}{'...' if len(failed_try)>10 else ''}"
        elif trial_score < (best_score - EPS_IMPROVE):
            shutil.copy(trial_csv, best_csv)
            best_score = trial_score
            per_n = per_n_try
            accepted = 1
        else:
            reason = "no_improve"

        rows.append({
            "iter": it,
            "n": target_n,
            "r": r,
            "rc": rc,
            "accepted": accepted,
            "best_score": best_score,
            "trial_score": trial_score,
            "fail_n": reason
        })

        if accepted:
            print(f"✅ iter {it:4d}: improved! n={target_n} r={r}  best={best_score:.12f}")
        elif it % 10 == 0:
            print(f"… iter {it:4d}: n={target_n} r={r}  best={best_score:.12f}  ({reason})")

        if it % CHECKPOINT_EVERY == 0:
            ck = f"best_checkpoint_iter{it:04d}.csv"
            shutil.copy(best_csv, ck)
            pd.DataFrame(rows).to_csv(log_csv, index=False)

    elapsed = time.time() - t0
    print(f"\n🏁 Done. Best score: {best_score:.12f}  (elapsed {elapsed:.1f}s)")
    shutil.copy(best_csv, "submission.csv")
    pd.DataFrame(rows).to_csv(log_csv, index=False)
    print("📌 Saved:")
    print(" - submission.csv (final)")
    print(" - best_submission.csv (best)")
    print(f" - {log_csv} (search log)")

main()
